[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/surface_metrics/glencoe_hillshade_slope.ipynb)

# Hillshade and slope maps of Glencoe

In this notebook you will:

1. Download a digital elevation model (DEM) of Glencoe in the Scottish Highlands with `lsdtt-fetch-raster`.
2. Compute a hillshade and a slope raster with `lsdtt-surface-metrics`.
3. Make two maps with `lsdviztools3`: a simple hillshade, and slope draped over the hillshade.

The `lsdtt3` programs are command-line tools controlled by small text **parameter files**. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools). This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# Install pygmt. mamba prints hundreds of lines, so we hide them and only
# show the end of the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3`, the plotting package, is on [PyPI](https://pypi.org/project/lsdviztools3/). It needs Python 3.12 or later, which Colab has. Version 0.4.2 or later is needed for the `lsdviz3-*` commands to run from a notebook.

In [ ]:
!pip install -q "lsdviztools3>=0.4.2"

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Tell the system where to find the `lsdtt3` programs and their libraries.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

Check that it works: this should print version information.

In [ ]:
!lsdtt-surface-metrics -v

## Step 1: Download a DEM of Glencoe

`lsdtt-fetch-raster` downloads Copernicus GLO-30 elevation data (30 m resolution, no account needed) for a box given in longitude and latitude.

It also **reprojects** the data into a metric coordinate system. Slope is a change in height divided by a horizontal distance, so both need to be in metres. Glencoe lies in UTM zone 30N (EPSG:32630). The tool would pick this zone automatically, but we set it explicitly.

The output is written to `<write_fname>_DEM.tif`, so here it will be `glencoe_DEM.tif`.

### Edit code block below for your site!

**Choosing the box with two right-clicks on Google Maps:**

1. On Google Maps, right-click the **lower left (south-west)** corner of your area. The first line of the menu that appears is the latitude and longitude, for example `55.9476273103734, -3.158220317286066`. Click it to copy it.
2. Paste it between the brackets after `lower_left =` (Ctrl-V, or Cmd-V on a Mac), so it reads `lower_left = (55.9476273103734, -3.158220317286066)`.
3. Do the same for the **upper right (north-east)** corner, after `upper_right =`.

Google Maps gives the latitude first, then the longitude.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (56.55, -5.25)
upper_right = (56.75, -4.75)

south, west = lower_left
north, east = upper_right


In [ ]:
# Parameter file for lsdtt-fetch-raster
with open("glencoe_fetch.param", "w") as f:
    f.write("write_fname: glencoe\n")
    f.write(f"west: {west}\n")
    f.write(f"east: {east}\n")
    f.write(f"south: {south}\n")
    f.write(f"north: {north}\n")
    f.write("dem_source: cop30_aws\n")
    f.write("target_epsg: 32630\n")
    f.write("grid_spacing: 30\n")

!lsdtt-fetch-raster ./ glencoe_fetch.param

In [ ]:
!ls -lh glencoe*

### Remove the sea

**Removing the sea.** The Copernicus DEM does not mark the sea as nodata: it stores it as an elevation of exactly **0 m**. If we leave it in, the sea is treated as a huge, perfectly flat surface. It shows up as a gradient of 0 on slope maps, swamps any histogram of gradients with a spurious spike at 0, and gives flow routing channels that wander across the sea. Glencoe has sea lochs (Loch Leven and Loch Linnhe), so this matters here.

`lsdtt-raster-preprocessing` with `min_elevation: 0` sets every pixel at or below 0 m to nodata, and writes the result to `glencoe_remove_seas.tif`. We store the name of the DEM to use in the variable `dem_fname`. (If you move the box inland, no pixels are at or below 0 m, no file is written, and `dem_fname` is just the original DEM.)

In [ ]:
import os

# Delete any masked DEM left over from an earlier run, so we never pick up a stale file
if os.path.exists(f"glencoe_remove_seas.tif"):
    os.remove(f"glencoe_remove_seas.tif")

with open(f"glencoe_remove_seas.param", "w") as f:
    f.write(f"read_fname: glencoe_DEM.tif\n")
    f.write(f"write_fname: glencoe\n")
    f.write("min_elevation: 0\n")      # pixels at or below 0 m become nodata

!lsdtt-raster-preprocessing ./ glencoe_remove_seas.param

# The masked DEM is only written if some pixels were at or below 0 m. An inland
# site has no sea, so then we simply carry on with the original DEM.
if os.path.exists(f"glencoe_remove_seas.tif"):
    dem_fname = f"glencoe_remove_seas.tif"
else:
    dem_fname = f"glencoe_DEM.tif"
print(f"\nUsing {dem_fname} from now on.")

## Step 2: Compute a hillshade and slope

`lsdtt-surface-metrics` calculates many properties of a surface. We ask for two:

- `write_hillshade_raster: true` gives a hillshade, a picture of the landscape lit by a sun in the northwest (azimuth 315°, 45° above the horizon). Output: `glencoe_hillshade.tif`.
- `polyfit_metrics: slope` fits a smooth surface to the elevations around each pixel and computes the topographic gradient from it (dimensionless, metres of drop per metre of horizontal distance, so 0.5 is a 1 in 2 slope, about 27°). `polyfit_lambda_metres` sets the size of the window used for the fit (70 m suits 30 m data). Output: `glencoe_direct_polyfit_slope.tif`.

We read `dem_fname`, the DEM with the sea removed. Note that `read_fname` needs the file extension.

In [ ]:
# Parameter file for lsdtt-surface-metrics
with open("glencoe_metrics.param", "w") as f:
    f.write(f"read_fname: {dem_fname}\n")
    f.write("write_fname: glencoe\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")
    f.write("polyfit_lambda_metres: 70\n")
    f.write("polyfit_metrics: slope\n")

!lsdtt-surface-metrics ./ glencoe_metrics.param

In [ ]:
!ls -lh glencoe*

## Step 3: A simple hillshade map

Now we plot the hillshade with `lsdviztools3`. `render_hillshade` draws a raster and returns a PyGMT figure.

- `shade=False` because our raster is already a hillshade (we don't want to shade it a second time).
- `MapStyle(clean=True)` removes all the map decoration (axes, labels, title) and adds a small black scale bar.
- `coordinates="local"` draws the map in UTM coordinates.

In [ ]:
from IPython.display import Image, display

from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_drape
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.base import save_figure

hs_style = MapStyle(cmap="gray", transparent=False, clean=True)
fig = render_hillshade(
    "glencoe_hillshade.tif",
    coordinates="local",
    shade=False,
    style=hs_style,
)
save_figure(fig, "glencoe_hillshade.png", style=hs_style)
display(Image("glencoe_hillshade.png", width=700))

## Step 4: Slope draped over the hillshade

`render_drape` draws a gray hillshade and then lays a second raster (here slope) on top in colour. `transparency` controls how much of the hillshade shows through (0 = solid colour, 100 = invisible).

This time we leave `clean` off, so the map has coordinates (in km) on the x and y axes. `colorbar=True` adds a colour bar.

**Question:** where are the steepest slopes? How do they relate to the valleys you can see in the hillshade?

In [ ]:
slope_style = MapStyle(cmap="jet", transparent=False)
fig = render_drape(
    "glencoe_hillshade.tif",                     # gray hillshade base
    "glencoe_direct_polyfit_slope.tif",  # coloured gradient layer
    coordinates="local",
    base_is_hillshade=True,
    transparency=40,
    cmap="jet",
    colorbar=True,
    colorbar_label="Gradient (m/m)",
    colorbar_position="right",
    style=slope_style,
)
save_figure(fig, "glencoe_slope.png", style=slope_style)
display(Image("glencoe_slope.png", width=700))

## Things to try

- Change `hillshade_azimuth` to move the sun (e.g. `45` for a northeast sun) and see how the hillshade changes.
- Change `polyfit_lambda_metres` to `150` and see how the slope map changes.
- Change the box to look at a different mountain area: right-click its two corners on Google Maps and paste them into `lower_left` and `upper_right`.

## Data

Always cite the data you use. This notebook downloads:

* **DEM:** Copernicus GLO-30, downloaded by `lsdtt-fetch-raster` (`dem_source: cop30_aws`). Cite it as: European Space Agency, Sinergise (2021). Copernicus Global Digital Elevation Model. Distributed by OpenTopography. https://doi.org/10.5069/G9028PQB. Licence notice: © DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018, provided under COPERNICUS by the European Union and ESA; all rights reserved.
